# NB65: Social Sentiment

Kafka -> Spark -> Redis/ES

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q "numpy<2.0.0" findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**
- **Elasticsearch**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start
# Start Elasticsearch
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.10.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.10.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.10.2
!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" ./elasticsearch-7.10.2/bin/elasticsearch -d > es.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9200) # Elasticsearch
wait_for_port(6379) # Redis


## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Tweets)

Simulates social media posts with hashtags.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Tweet Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 500 tweets...")
tags = ['#happy', '#sad', '#neutral']
for _ in range(500):
    data = {'text': f'I am feeling {random.choice(tags)}', 'tag': random.choice(tags)}
    producer.send('input-topic', json.dumps(data).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Sentiment Dashboard

1. Aggregates counts by tag locally.
2. Updates Redis counters (Hash `sentiment_counts`).

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis, json

spark = SparkSession.builder.appName("Social").getOrCreate()

def process_batch(df, epoch_id):
    data = [json.loads(r.value) for r in df.collect()]
    if not data: return
    r = redis.Redis()
    for d in data:
        r.hincrby("sentiment_counts", d['tag'], 1)
    print(f"Batch {epoch_id} updated counts in Redis.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Read sentiment counts from Redis.

In [ ]:
import redis
r = redis.Redis()
print(r.hgetall("sentiment_counts"))